In [ ]:
from collections import Counter

import pandas as pd
import tldextract
from forestplot import forestplot
import matplotlib.pyplot as plt
plt.rcParams['font.monospace'] = ['Noto Mono', 'Tlwg Mono', 'Courier New']
from matplotlib.ticker import FuncFormatter
from tqdm.notebook import tqdm
from urlextract import URLExtract

from helpers import categorize_domain, save_mpl_fig

url_extractor = URLExtract()

df = pd.read_parquet("gc_human_tweets.parquet")
df

## Top domains

In [ ]:
# Extract URLs and domains from all tweets
all_urls = []
all_domains = []
tweet_domain_mapping = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    content = row["content"]
    if pd.notna(content):
        urls = url_extractor.find_urls(str(content))
        if urls:
            all_urls.extend(urls)

            domains_in_tweet = set()  # Track unique domains per tweet

            # Extract domain from each URL
            for url in urls:
                extracted = tldextract.extract(url)
                # Combine domain + suffix (e.g., 'nytimes' + 'com' = 'nytimes.com')
                if extracted.domain and extracted.suffix:
                    domain = f"{extracted.domain}.{extracted.suffix}".lower()
                    all_domains.append(domain)
                    domains_in_tweet.add(domain)

            # Store tweet-level data for each unique domain in this tweet
            for domain in domains_in_tweet:
                tweet_domain_mapping.append(
                    {
                        "domain": domain,
                        "tweet_idx": idx,
                        "retweets": row["retweets"],
                        "favorites": row["favorites"],
                        "n_hashtags": row["n_hashtags"],
                    }
                )

df_domains = (
    pd.DataFrame.from_dict(Counter(all_domains), orient="index", columns=["count"])
    .sort_values("count", ascending=False)
    .reset_index(names="domain")
)
df_domains.to_csv("output/domains.csv", index=False)

# Create tweet-domain mapping for metrics
df_tweet_domains = pd.DataFrame(tweet_domain_mapping)

df_domains

## Plot

### prep

In [ ]:
TOP_N = 30
df_plot = df_domains[df_domains["domain"] != "twitter.com"].head(TOP_N).copy()

# Calculate percentage of URLs
total_urls = df_domains["count"].sum()
df_plot["pct"] = (df_plot["count"] / total_urls) * 100

# Add metrics from tweet mapping
for idx, row in df_plot.iterrows():
    domain = row["domain"]
    tweets_with_domain = df_tweet_domains[df_tweet_domains["domain"] == domain]

    df_plot.loc[idx, "mean_retweets"] = tweets_with_domain["retweets"].mean()
    df_plot.loc[idx, "mean_favs"] = tweets_with_domain["favorites"].mean()
    df_plot.loc[idx, "mean_hashtags"] = tweets_with_domain["n_hashtags"].mean()
    df_plot.loc[idx, "category"] = categorize_domain(domain)

df_plot["count_str"] = df_plot["count"].apply(lambda x: f"{x:,}".rjust(6))
df_plot["pct_str"] = df_plot["pct"].apply(lambda x: f"{x:.1f}%".rjust(6))
df_plot["rt_str"] = df_plot["mean_retweets"].apply(lambda x: f"{x:.1f}".rjust(5))
df_plot["fav_str"] = df_plot["mean_favs"].apply(lambda x: f"{x:.1f}".rjust(4))
df_plot["cat_str"] = df_plot["category"].str.rjust(9)

# For forestplot - use pct as both estimate and limits (creates dot plot)
df_plot["ll"] = df_plot["pct"]
df_plot["hl"] = df_plot["pct"]
df_plot

### plot

In [ ]:
# plot
ax = forestplot(
    df_plot,
    estimate="pct",
    ll="ll",
    hl="hl",
    varlabel="domain",
    annote=[
        "cat_str",
        "rt_str",
        "fav_str",
    ],  # Left annotations: category, retweets, favorites
    annoteheaders=["Category", "   RT", "Fave"],
    rightannote=["count_str", "pct_str"],
    right_annoteheaders=["  URLs", "URLs(%)"],
    ci_report=False,
    figsize=(5, TOP_N * 0.4),  # Slightly wider for category column
    color_alt_rows=True,
    table=True,
    xticks=range(0, 6),
    **{
        "marker": "o",
        "variable_header": "Domain",
        "markersize": 40,
        "xlinecolor": ".8",
        "xtick_size": 12,
    },
)

# Format x-axis as percentages
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f"{int(x)}%"))
ax.set_xlabel("Percentage of URLs", fontsize=12)

# Add subtle gridlines
for vline in range(0, 6):
    ax.vlines(vline, ymin=-0.5, ymax=TOP_N - 0.5, colors=".85", linewidth=0.5)

save_mpl_fig("output/figures/top_domains_tabulation")